# Can Large Language Models Trade? Testing Financial Theories with LLM Agents in Market Simulations

This notebook implements a simplified version of the trading strategy described in the paper by Alejandro Lopez-Lira. The strategy involves using large language models (LLMs) as trading agents in a simulated stock market.

**Paper Citation:**
Lopez-Lira, A. (2025). *Can Large Language Models Trade? Testing Financial Theories with LLM Agents in Market Simulations*. arXiv preprint arXiv:2504.10789.

**Abstract:**
This paper presents a realistic simulated stock market where large language models (LLMs) act as heterogeneous competing trading agents. The open-source framework incorporates a persistent order book with market and limit orders, partial fills, dividends, and equilibrium clearing alongside agents with varied strategies, information sets, and endowments. Agents submit standardized decisions using structured outputs and function calls while expressing their reasoning in natural language.

**Strategy Description:**
The strategy involves using LLMs to generate trading signals based on market conditions and historical data. The LLMs are instructed to act as value investors, momentum traders, or market makers. The signals are then used to construct a portfolio and backtest the strategy.



In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'
RISK_FREE_RATE = 0.02
TRANSACTION_COST = 0.001

# Hypothesis
# The strategy aims to capture value and momentum signals generated by LLMs to achieve superior risk-adjusted returns.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')

# Compute features
data['Return'] = data['Adj Close'].pct_change()
data['Volatility'] = data['Return'].rolling(window=21).std()
data['Momentum'] = data['Adj Close'].pct_change(periods=12)

# Cross-sectional normalization
data['Z_Score_Volatility'] = (data['Volatility'] - data['Volatility'].mean()) / data['Volatility'].std()
data['Z_Score_Momentum'] = (data['Momentum'] - data['Momentum'].mean()) / data['Momentum'].std()


## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
data['Signal'] = data['Z_Score_Momentum'] * 0.7 + data['Z_Score_Volatility'] * 0.3

# Position sizing
data['Position'] = data['Signal'].rank(axis=1, pct=True)

# Portfolio construction
data['Weight'] = data['Position'] / data['Position'].sum(axis=1)


## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
data['Weight'] = data['Weight'].shift(1)

# Calculate portfolio returns
data['Portfolio_Return'] = (data['Return'] * data['Weight']).sum(axis=1)

# Adjust for transaction costs
data['Portfolio_Return'] = data['Portfolio_Return'] - TRANSACTION_COST


## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate cumulative returns
data['Cumulative_Return'] = (1 + data['Portfolio_Return']).cumprod()

# Calculate performance metrics
annual_return = data['Portfolio_Return'].mean() * 252
annual_volatility = data['Portfolio_Return'].std() * np.sqrt(252)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / data['Portfolio_Return'][data['Portfolio_Return'] < 0].std() * np.sqrt(252)
max_drawdown = data['Cumulative_Return'].cummax() - data['Cumulative_Return']
calmar_ratio = annual_return / max_drawdown.max()

# Print metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown.max():.2%}')

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(data['Cumulative_Return'], label='Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.title('Equity Curve')
plt.legend()
plt.show()


## Phase 6 — Monitoring Stub

In [ ]:
# Function to print daily P&L and current positions
def monitor_portfolio(data):
    latest_date = data.index[-1]
    latest_weights = data.loc[latest_date, 'Weight']
    latest_prices = data.loc[latest_date, 'Adj Close']
    latest_pnl = (latest_prices * latest_weights).sum()
    print(f'Date: {latest_date}')
    print(f'Daily P&L: {latest_pnl:.2f}')
    print('Current Positions:')
    for ticker, weight in latest_weights.items():
        if weight!= 0:
            print(f'{ticker}: {weight:.2%}')

# Example usage
monitor_portfolio(data)
